# 模块二：SwellEx-96 GPU 网络训练、预测与实验结果分析

本 Notebook 专为 **Google Colab GPU 运行时**（`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`）设计，直接消费 **模块一（`01_data_processing_cpu.ipynb`，原始数据位于 `MyDrive/Data/OpenAccess/SwellEx`）** 预先保存在 `/content/drive/MyDrive/Data/SwellEx_Colab/datasets/` 中的独立配置 HDF5 数据集，**无需再次读取 `Origindata/`，也不会重复执行任何 CPU 频域特征提取**。

### 高效 I/O 与实验独立性策略
1. **自动引导仓库**：Colab 会话启动后先挂载 Google Drive，自动克隆或同步最新 `SwellEx` 到 `/content/SwellEx`。
2. **按需本地 NVMe 缓存**：根据 `dataset_manifest.json` 中指定的独立 `DATASET_CONFIG_KEY`，仅将本次 `METHODS` 所需的 HDF5 目录复制到实例本地高速盘 `/content/local_datasets/`，避免多轮 `DataLoader` 反复穿透 Google Drive FUSE。
3. **全参数化与结果完全隔离**：完整加载 TOML `[training]`（含 `val_split_mode = "random" | "blocked"`、`val_gap_s` 等）与 `[grnn]` 参数并做严格 schema 校验；每次实验组合在 `/content/drive/MyDrive/Data/SwellEx_Colab/experiments/<run_id>/` 下创建独立目录并写入 `run_manifest.json`，不同数据集或超参组合绝不互相覆盖。

In [ ]:
# Cell 1: Google Drive 挂载、仓库自动引导、GPU 环境检查与 dataset_manifest.json 独立配置读取
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# --- 与模块一保持一致的 Google Drive 工作区配置 ---
DRIVE_WORKSPACE_SUBDIR = "Data/SwellEx_Colab"  # 对应 /content/drive/MyDrive/Data/SwellEx_Colab
ORIGINDATA_DRIVE_URL = "https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link"
REPO_GIT_URL = "https://github.com/AmazingUmi/SwellEx.git"
REPO_GIT_BRANCH = "main"
AUTO_UPDATE_REPO = True     # Colab 下每次运行 Cell 1 自动向 GitHub 拉取并同步最新提交
FORCE_RECLONE_REPO = False  # 若想彻底删除 /content/SwellEx 重新克隆，设为 True 运行一次即可
# ------------------------------------------------

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)

DRIVE_MOUNT_ROOT = Path("/content/drive")
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)


def get_git_commit_info(repo_dir: Path) -> tuple[str, str]:
    proc = subprocess.run(
        ["git", "-C", str(repo_dir), "log", "-1", "--format=%h|%s"],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and "|" in proc.stdout:
        short_sha, subject = proc.stdout.strip().split("|", 1)
        return short_sha, subject
    return "unknown", ""


def reload_project_modules() -> None:
    for mod_name in [m for m in list(sys.modules) if m == "scripts" or m.startswith("scripts.")]:
        sys.modules.pop(mod_name, None)
    importlib.invalidate_caches()


def ensure_project_root() -> Path:
    if not IN_COLAB:
        for candidate in (Path.cwd(), Path.cwd().parent):
            if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
                return candidate.resolve()
        raise FileNotFoundError("未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。")

    colab_repo = Path("/content/SwellEx")
    if FORCE_RECLONE_REPO and colab_repo.exists():
        shutil.rmtree(colab_repo)
        print(f"🗑️ 已按 FORCE_RECLONE_REPO=True 清除旧目录: {colab_repo}")

    if (colab_repo / ".git").is_dir() and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        old_sha, _ = get_git_commit_info(colab_repo)
        if AUTO_UPDATE_REPO:
            subprocess.run(["git", "-C", str(colab_repo), "remote", "set-url", "origin", REPO_GIT_URL], capture_output=True)
            fetch_proc = subprocess.run(
                ["git", "-C", str(colab_repo), "fetch", "--depth", "1", "origin", REPO_GIT_BRANCH],
                capture_output=True,
                text=True,
            )
            if fetch_proc.returncode == 0:
                subprocess.run(["git", "-C", str(colab_repo), "reset", "--hard", "FETCH_HEAD"], check=True, capture_output=True)
                subprocess.run(
                    ["git", "-C", str(colab_repo), "clean", "-fd", "-e", "outputs", "-e", "output", "-e", "Origindata"],
                    capture_output=True,
                )
                new_sha, subject = get_git_commit_info(colab_repo)
                if old_sha != new_sha:
                    print(f"🔄 已同步云端仓库最新提交: {old_sha} -> {new_sha} ({subject})")
                else:
                    print(f"✅ 云端仓库已是最新提交: {new_sha} ({subject})")
            else:
                print(f"⚠️ 检查远端更新失败（继续使用现有副本 {old_sha}）: {fetch_proc.stderr.strip()}")
        return colab_repo.resolve()

    if colab_repo.exists():
        shutil.rmtree(colab_repo)
    print(f"📥 正在克隆最新云端仓库 {REPO_GIT_URL} ({REPO_GIT_BRANCH}) -> {colab_repo} ...")
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_GIT_BRANCH, REPO_GIT_URL, str(colab_repo)],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        new_sha, subject = get_git_commit_info(colab_repo)
        print(f"✅ 克隆完成: {new_sha} ({subject})")
        return colab_repo.resolve()

    fallback_candidates = [Path.cwd(), Path.cwd().parent]
    if DRIVE_MOUNT_ROOT.exists():
        fallback_candidates.extend([
            DRIVE_MOUNT_ROOT / "MyDrive" / "SwellEx",
            DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "SwellEx",
        ])
    for candidate in fallback_candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            print(f"⚠️ GitHub 克隆未成功，已回退使用挂载目录副本: {candidate.resolve()}")
            return candidate.resolve()
    raise FileNotFoundError(
        f"自动克隆仓库失败（可能仓库为私有）: {proc.stderr.strip()}\n"
        "请先将最新 SwellEx 仓库上传至 /content/drive/MyDrive/SwellEx 或使用带权限链接克隆。"
    )


PROJECT_ROOT = ensure_project_root()
reload_project_modules()
REPO_COMMIT, REPO_SUBJECT = get_git_commit_info(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    DRIVE_BASE = DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR
    if not (DRIVE_BASE / "datasets" / "dataset_manifest.json").exists():
        for alt_sub in ("Data", "SwellEx_Colab"):
            alt_base = DRIVE_MOUNT_ROOT / "MyDrive" / alt_sub
            if (alt_base / "datasets" / "dataset_manifest.json").exists():
                DRIVE_BASE = alt_base
                break
    LOCAL_DATASET_CACHE = Path("/content/local_datasets")
else:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))
    LOCAL_DATASET_CACHE = PROJECT_ROOT / "outputs" / "local_dataset_cache"

DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"
DRIVE_EXP_ROOT = DRIVE_BASE / "experiments"
MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"

HAS_TORCH = importlib.util.find_spec("torch") is not None
CUDA_AVAILABLE = False
if HAS_TORCH:
    import torch
    CUDA_AVAILABLE = bool(torch.cuda.is_available())
    if CUDA_AVAILABLE:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ CUDA GPU 已就绪: {gpu_name} ({vram_gb:.1f} GiB), PyTorch {torch.__version__}")
    else:
        print("⚠️ 未检测到可用 CUDA GPU！在 Colab 中请切换 Runtime -> Change runtime type -> GPU。")
else:
    print("ℹ️ 当前本地解释器未安装 PyTorch（将以配置校验/历史结果分析模式运行；Colab GPU 环境自带 PyTorch）。")

MANIFEST = (
    json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if MANIFEST_PATH.exists()
    else {"schema_version": 2, "configs": {}}
)


def collect_manifest_configs(manifest: dict) -> dict[str, dict]:
    """读取 schema v2 的独立 configs 表，并兼容旧版 presets 结构（只读转换，不混写）。"""
    configs: dict[str, dict] = {}
    for key, item in manifest.get("configs", {}).items():
        configs[key] = item
    for legacy_key, item in manifest.get("presets", {}).items():
        if legacy_key not in configs and isinstance(item, dict) and "base_routes" in item:
            configs[legacy_key] = {
                "config_key": legacy_key,
                "preset": legacy_key.split("_smoke_")[0],
                "methods": list(item["base_routes"].keys()),
                "routes": item["base_routes"],
                "legacy_manifest_entry": True,
            }
    return configs


AVAILABLE_CONFIGS = collect_manifest_configs(MANIFEST)
origin_prov = MANIFEST.get("origindata_source", {})
manifest_commit = MANIFEST.get("repo_commit")
print(f"PROJECT_ROOT       : {PROJECT_ROOT} (commit={REPO_COMMIT})")
print(f"MANIFEST_PATH      : {MANIFEST_PATH} (存在={MANIFEST_PATH.exists()}, 数据生成 commit={manifest_commit or '未知'})")
if origin_prov:
    print(f"原始数据溯源 (只读): FolderID={origin_prov.get('drive_folder_id')} ({origin_prov.get('resolution_mode')})")
elif not MANIFEST_PATH.exists():
    print(f"ℹ️ 尚未发现 dataset_manifest.json：请先在 CPU 运行时执行 01_data_processing_cpu.ipynb（已绑定 {ORIGINDATA_DRIVE_URL}）。")

print(f"实验结果保存根目录 : {DRIVE_EXP_ROOT}")
print(f"可用独立数据集配置 : 共 {len(AVAILABLE_CONFIGS)} 套 (最近更新: {MANIFEST.get('latest_config_key', '无')})")
for cfg_key, cfg_info in AVAILABLE_CONFIGS.items():
    gen_cfg = cfg_info.get("generation_config", {})
    d_cfg = gen_cfg.get("dataset", {})
    s_cfg = gen_cfg.get("scm", {})
    scm_desc = f"Ns={s_cfg.get('num_snapshots_per_segment', '?')},ov={s_cfg.get('snapshot_overlap_count', '?')}" if "SCM" in cfg_info.get("methods", []) and s_cfg else "-"
    print(
        f"  - {cfg_key} | methods={cfg_info.get('methods')} | "
        f"split={d_cfg.get('split_strategy', 'legacy')} | SCM({scm_desc}) | "
        f"win={d_cfg.get('fft_window', '?')}/{d_cfg.get('signal_centering', '?')}"
    )

## 1. `base_pack` 注册表与预设 TOML 速览

下方单元格调用仓库纯标准库模块（`scripts.remote_workflow.packs`），确认当前唯一激活的实验包 `base_pack` 支持的合法 `METHODS`（`RBD` / `SCM`）与 `NETWORKS`（`complex_CNN` / `ResNet18` / `ResNet50` / `GRNN`）。

In [ ]:
# Cell 2: 查看 base_pack 支持的特征路由/网络组合与官方 TOML 预设
from scripts.remote_workflow.packs import DEFAULT_PACK, PACKS

print(f"=== 实验包注册表 (DEFAULT_PACK = {DEFAULT_PACK!r}) ===")
for name, pack in PACKS.items():
    print(f"- {name:10s}: methods={list(pack.methods)} | networks={list(pack.networks)}")

presets_dir = PROJECT_ROOT / "scripts" / "remote_workflow" / "presets"
print("\n=== 官方 TOML 预设文件 ===")
for toml_file in sorted(presets_dir.glob("*.toml")):
    print(f"- {toml_file.stem:14s}: {toml_file.relative_to(PROJECT_ROOT)}")

## 2. 神经网络与独立 GRNN 训练 / 预测运行器（全参数化 & 结果独立隔离）

- **显式选择数据集与子集**：
  - `DATASET_CONFIG_KEY`：对应模块一同步到 `dataset_manifest.json` 的 `CONFIG_KEY`（当云端存在多套数据集配置时**必须显式填写**，杜绝跨配置混淆）；
  - `METHODS` 与 `NETWORKS`：**均必须显式指定且数目 $\ge 1$**（例如多快拍协方差矩阵配合复数卷积网络设 `METHODS = ["SCM"]`、`NETWORKS = ["complex_CNN"]`）。
- **完整转发 TOML `[training]` / `[grnn]` 参数**：
  - 支持 `TRAINING_OVERRIDES = {"val_split_mode": "blocked", "val_gap_s": 6.0, ...}` 阻断重叠多快拍窗口在训练/验证集之间的时间相关性泄漏；
  - 支持 `GRNN_OVERRIDES`（含 `cv_folds = 0` 直接使用固定 `spread`）。
- **独立实验产出**：每次执行自动生成规范运行目录 `DRIVE_EXP_ROOT/<run_id>/`，显式绑定刚生成的 `--checkpoint` / `--reference` 执行 `predict`，并落盘包含完整参数快照与测试集指标的 `run_manifest.json`。

In [ ]:
# Cell 3: base_pack 全参数 GPU 训练与测试预测（每套配置独立落盘）
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json
import subprocess

from scripts.network_training.catalog import entry_script as select_entry
from scripts.remote_workflow.config import load_config, validate_config
from scripts.remote_workflow.metrics_contract import (
    compute_prediction_metrics,
    parse_prediction_file,
)
from scripts.remote_workflow.model_plan import model_plan
from scripts.remote_workflow.packs import DEFAULT_PACK
from scripts.remote_workflow.run_identity import make_run_id, validate_filters

# --- 用户实验参数（按需修改） ---
RUN_TRAINING = False               # 在 Colab GPU 就绪且确认下方预览无误后设为 True
DATASET_CONFIG_KEY = None          # 填入模块一输出的 CONFIG_KEY；若云端仅 1 套配置可留 None 自动绑定
PRESET_TOML = "train_before"       # 基础训练超参预设: "train_before" | "train_after" | "full"
CUSTOM_CONFIG_TOML = None          # 若指定自定义 TOML 路径，优先于 PRESET_TOML

METHODS = ["SCM"]                  # 必须指定，数目 >= 1，如 ["SCM"] 或 ["RBD", "SCM"]
NETWORKS = ["complex_CNN"]         # 必须指定，数目 >= 1，如 ["complex_CNN", "ResNet18", "ResNet50", "GRNN"]
SEED_OVERRIDE = None               # None 则使用 TOML [validation].seed

# 按需覆盖 TOML [training] 或 [grnn] 字段（多快拍重叠实验强烈建议启用 blocked 验证切分）：
# 示例: TRAINING_OVERRIDES = {"val_split_mode": "blocked", "val_gap_s": 6.0, "epochs": 100}
TRAINING_OVERRIDES: dict = {}
GRNN_OVERRIDES: dict = {}
# -------------------------------

PRESET_TOML_MAP = {
    "train_before": PROJECT_ROOT / "scripts/remote_workflow/presets/train_before.toml",
    "train_after": PROJECT_ROOT / "scripts/remote_workflow/presets/train_after.toml",
    "full": PROJECT_ROOT / "scripts/remote_workflow/presets/full.toml",
}

if CUSTOM_CONFIG_TOML is not None:
    toml_path = Path(CUSTOM_CONFIG_TOML).expanduser().resolve()
else:
    if PRESET_TOML not in PRESET_TOML_MAP:
        raise ValueError(f"未知 PRESET_TOML={PRESET_TOML!r}，可选: {list(PRESET_TOML_MAP)}")
    toml_path = PRESET_TOML_MAP[PRESET_TOML]

if not isinstance(METHODS, (list, tuple)) or len(METHODS) < 1:
    raise ValueError("METHODS 必须显式指定为非空列表（数目最小为 1）。")
if not isinstance(NETWORKS, (list, tuple)) or len(NETWORKS) < 1:
    raise ValueError("NETWORKS 必须显式指定为非空列表（数目最小为 1）。")

selection_filter = validate_filters(
    DEFAULT_PACK,
    {"methods": list(METHODS), "networks": list(NETWORKS)},
)
planned_jobs = model_plan("selected", selection_filter)

effective_cfg = load_config(toml_path)
effective_cfg["experiment"]["methods"] = list(selection_filter["methods"])
effective_cfg["experiment"]["networks"] = list(selection_filter["networks"])
if SEED_OVERRIDE is not None:
    effective_cfg["validation"]["seed"] = int(SEED_OVERRIDE)
for key, val in TRAINING_OVERRIDES.items():
    if key not in effective_cfg["training"]:
        raise ValueError(f"[training] 未知覆盖字段: {key!r}")
    effective_cfg["training"][key] = deepcopy(val)
for key, val in GRNN_OVERRIDES.items():
    if key not in effective_cfg["grnn"]:
        raise ValueError(f"[grnn] 未知覆盖字段: {key!r}")
    effective_cfg["grnn"][key] = deepcopy(val)

# 解析目标数据集 CONFIG_KEY（拒绝多配置下的隐式猜测）
resolved_dataset_key = DATASET_CONFIG_KEY
if resolved_dataset_key is None and len(AVAILABLE_CONFIGS) == 1:
    resolved_dataset_key = next(iter(AVAILABLE_CONFIGS))

selected_dataset_entry = AVAILABLE_CONFIGS.get(resolved_dataset_key) if resolved_dataset_key else None
if selected_dataset_entry and "generation_config" in selected_dataset_entry:
    gen_snap = selected_dataset_entry["generation_config"]
    effective_cfg["dataset"] = deepcopy(gen_snap["dataset"])
    effective_cfg["rbd"] = deepcopy(gen_snap["rbd"])
    effective_cfg["scm"] = deepcopy(gen_snap["scm"])

effective_cfg = validate_config(effective_cfg)


def build_neural_commands(
    entry_script_path: Path,
    method: str,
    model_id: str,
    data_dir: str,
    method_out_dir: str,
    checkpoint_placeholder: str,
    cfg: dict,
) -> tuple[list[str], list[str]]:
    t, v = cfg["training"], cfg["validation"]
    train_cmd = [
        sys.executable, "-B", str(entry_script_path), "train",
        "--method", method,
        "--model", model_id,
        "--data", data_dir,
        "--output-dir", method_out_dir,
        "--no-resume",
        "--epochs", str(t["epochs"]),
        "--batch-size", str(t["batch_size"]),
        "--lr", str(t["lr"]),
        "--weight-decay", str(t["weight_decay"]),
        "--base-channels", str(t["base_channels"]),
        "--dropout", str(t["dropout"]),
        "--loss-space", t["loss_space"],
        "--huber-beta", str(t["huber_beta"]),
        "--val-split-mode", t["val_split_mode"],
        "--val-fraction", str(t["val_fraction"]),
        "--val-gap-s", str(t["val_gap_s"]),
        "--seed", str(v["seed"]),
        "--num-workers", str(t["num_workers"]),
        "--device", "cuda",
    ]
    if not t["normalize_input"]:
        train_cmd.append("--no-input-norm")

    predict_cmd = [
        sys.executable, "-B", str(entry_script_path), "predict",
        "--method", method,
        "--model", model_id,
        "--data", data_dir,
        "--output-dir", method_out_dir,
        "--checkpoint", checkpoint_placeholder,
        "--batch-size", str(t["predict_batch_size"]),
        "--num-workers", str(t["num_workers"]),
        "--device", "cuda",
        "--print-first", "0",
    ]
    return train_cmd, predict_cmd


def build_grnn_commands(
    entry_script_path: Path,
    method: str,
    data_dir: str,
    method_out_dir: str,
    reference_placeholder: str,
    cfg: dict,
) -> tuple[list[str], list[str]]:
    g, v = cfg["grnn"], cfg["validation"]
    build_cmd = [
        sys.executable, "-B", str(entry_script_path), "build",
        "--method", method,
        "--data", data_dir,
        "--output-dir", method_out_dir,
        "--cv-folds", str(g["cv_folds"]),
        "--spread", str(g["spread"]),
    ]
    if g["sigma_candidates"]:
        build_cmd.extend(["--sigma-candidates", *map(str, g["sigma_candidates"])])
    build_cmd.extend([
        "--seed", str(v["seed"]),
        "--batch-size", str(g["batch_size"]),
        "--device", "cuda",
    ])
    if g["standardize_input"]:
        build_cmd.append("--standardize-input")
    if g["per_sample_channel_norm"]:
        build_cmd.append("--per-sample-channel-norm")
    if g["max_reference_samples"] > 0:
        build_cmd.extend(["--max-reference-samples", str(g["max_reference_samples"])])
    if g["max_validation_samples"] > 0:
        build_cmd.extend(["--max-validation-samples", str(g["max_validation_samples"])])

    predict_cmd = [
        sys.executable, "-B", str(entry_script_path), "predict",
        "--method", method,
        "--data", data_dir,
        "--output-dir", method_out_dir,
        "--reference", reference_placeholder,
        "--batch-size", str(g["predict_batch_size"]),
        "--device", "cuda",
        "--print-first", "0",
    ]
    if g["max_predict_samples"] > 0:
        predict_cmd.extend(["--max-predict-samples", str(g["max_predict_samples"])])
    return build_cmd, predict_cmd


def stage_h5_from_drive(config_key: str, method: str) -> Path:
    cfg_entry = AVAILABLE_CONFIGS.get(config_key)
    if not cfg_entry:
        raise KeyError(
            f"dataset_manifest.json 中未找到 DATASET_CONFIG_KEY={config_key!r}，当前可用: {list(AVAILABLE_CONFIGS)}"
        )
    routes = cfg_entry.get("routes", {})
    if method not in routes:
        raise KeyError(
            f"数据集配置 {config_key!r} 未包含方法 {method!r}（该配置仅生成了 {list(routes)}）。"
        )
    rel_dir = Path(routes[method]["rel_dir"])
    src_dir = DRIVE_DATASET_ROOT / rel_dir
    if not src_dir.is_dir():
        raise FileNotFoundError(f"Google Drive 数据集目录不存在: {src_dir}")
    dst_dir = LOCAL_DATASET_CACHE / rel_dir
    src_files = sorted(p for p in src_dir.iterdir() if p.is_file() and not p.name.startswith("._"))
    cache_ready = (
        dst_dir.is_dir()
        and all((dst_dir / p.name).is_file() and (dst_dir / p.name).stat().st_size == p.stat().st_size for p in src_files)
    )
    if not cache_ready:
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        dst_dir.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_dir, dst_dir)
    return dst_dir


print(f"基础配置 TOML    : {toml_path.relative_to(PROJECT_ROOT) if toml_path.is_relative_to(PROJECT_ROOT) else toml_path}")
print(f"目标数据集配置   : {resolved_dataset_key or '<待指定 DATASET_CONFIG_KEY>'}")
print(f"执行任务计划     : 共 {len(planned_jobs)} 项 -> {[(j['method'], j['architecture']) for j in planned_jobs]}")
print(
    f"训练切分与优化   : epochs={effective_cfg['training']['epochs']}, batch={effective_cfg['training']['batch_size']}, "
    f"lr={effective_cfg['training']['lr']}, val_split={effective_cfg['training']['val_split_mode']} "
    f"(frac={effective_cfg['training']['val_fraction']}, gap={effective_cfg['training']['val_gap_s']}s)"
)

if not RUN_TRAINING:
    print("ℹ️ 当前 RUN_TRAINING=False（预览模式）。确认数据集与参数无误后将 RUN_TRAINING 设为 True 即可启动 GPU 训练。")
    if len(AVAILABLE_CONFIGS) > 1 and DATASET_CONFIG_KEY is None:
        print(f"⚠️ 云端存在多套独立数据集 {list(AVAILABLE_CONFIGS)}，启动前请显式填写 DATASET_CONFIG_KEY。")
else:
    if not CUDA_AVAILABLE:
        raise RuntimeError("拒绝静默回退到 CPU：请先将 Colab 运行时切换为 GPU！")
    if resolved_dataset_key is None:
        if not AVAILABLE_CONFIGS:
            raise FileNotFoundError("dataset_manifest.json 中尚无任何可用数据集，请先运行模块一生成数据。")
        raise ValueError(
            f"检测到多套独立数据集配置 {list(AVAILABLE_CONFIGS)}，请显式设置 DATASET_CONFIG_KEY 以保证各配置实验严格独立。"
        )
    if resolved_dataset_key not in AVAILABLE_CONFIGS:
        raise KeyError(f"未知 DATASET_CONFIG_KEY={resolved_dataset_key!r}，可选: {list(AVAILABLE_CONFIGS)}")

    required_methods = sorted({j["method"] for j in planned_jobs})
    staged_dirs = {m: stage_h5_from_drive(resolved_dataset_key, m) for m in required_methods}

    run_id = make_run_id(DEFAULT_PACK, effective_cfg, selection_filter)
    run_dir = DRIVE_EXP_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=False)

    config_sha256 = hashlib.sha256(
        json.dumps(
            {"dataset_config_key": resolved_dataset_key, "effective_config": effective_cfg},
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()

    run_manifest = {
        "schema_version": 1,
        "pack": DEFAULT_PACK,
        "run_id": run_id,
        "status": "RUNNING",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "repo_commit": REPO_COMMIT,
        "dataset_config_key": resolved_dataset_key,
        "config_sha256": config_sha256,
        "selection": selection_filter,
        "effective_config": effective_cfg,
        "jobs": [],
    }
    run_manifest_path = run_dir / "run_manifest.json"
    run_manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    print(f"💾 独立实验输出目录: {run_dir}")
    try:
        for idx, job in enumerate(planned_jobs, 1):
            method = job["method"]
            arch = job["architecture"]
            model_id = job["model"]
            entry_path = PROJECT_ROOT / select_entry(job)
            local_data_dir = staged_dirs[method]
            method_out_dir = run_dir / "networks_results" / method
            method_out_dir.mkdir(parents=True, exist_ok=True)

            print(f"\n=== [{idx}/{len(planned_jobs)}] {method} + {arch} ({model_id}) ===")
            if arch == "GRNN":
                build_cmd, _ = build_grnn_commands(
                    entry_path, method, str(local_data_dir), str(method_out_dir), "<PENDING>", effective_cfg
                )
                print(f">>> build: {' '.join(build_cmd)}")
                subprocess.run(build_cmd, cwd=PROJECT_ROOT, check=True)
                refs = sorted((method_out_dir / model_id / "reference_outputs").glob("*_reference_*.pt"))
                if len(refs) != 1:
                    raise RuntimeError(f"期望生成唯一 GRNN reference 文件，实际发现: {refs}")
                _, pred_cmd = build_grnn_commands(
                    entry_path, method, str(local_data_dir), str(method_out_dir), str(refs[0]), effective_cfg
                )
                print(f">>> predict: {' '.join(pred_cmd)}")
                subprocess.run(pred_cmd, cwd=PROJECT_ROOT, check=True)
                artifact_path = str(refs[0].relative_to(run_dir))
            else:
                train_cmd, _ = build_neural_commands(
                    entry_path, method, model_id, str(local_data_dir), str(method_out_dir), "<PENDING>", effective_cfg
                )
                print(f">>> train: {' '.join(train_cmd)}")
                subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)
                ckpts = sorted((method_out_dir / model_id / "train_outputs").glob("*_best_*.pt"))
                if len(ckpts) != 1:
                    raise RuntimeError(f"期望生成唯一 best checkpoint，实际发现: {ckpts}")
                _, pred_cmd = build_neural_commands(
                    entry_path, method, model_id, str(local_data_dir), str(method_out_dir), str(ckpts[0]), effective_cfg
                )
                print(f">>> predict: {' '.join(pred_cmd)}")
                subprocess.run(pred_cmd, cwd=PROJECT_ROOT, check=True)
                artifact_path = str(ckpts[0].relative_to(run_dir))

            pred_csvs = sorted((method_out_dir / model_id / "test_outputs").glob("*_predictions_*.csv"))
            if len(pred_csvs) != 1:
                raise RuntimeError(f"期望生成唯一预测 CSV，实际发现: {pred_csvs}")
            metrics = compute_prediction_metrics(parse_prediction_file(pred_csvs[0]))
            metrics.pop("source_segment_indices", None)
            run_manifest["jobs"].append({
                "method": method,
                "architecture": arch,
                "model": model_id,
                "artifact": artifact_path,
                "prediction_csv": str(pred_csvs[0].relative_to(run_dir)),
                "metrics": metrics,
            })
            run_manifest_path.write_text(
                json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
            )
            print(
                f"✅ {method}+{arch}: RMSE={metrics['rmse_km']:.4f} km | MAE={metrics['mae_km']:.4f} km | "
                f"Bias={metrics['bias_km']:+.4f} km | DebiasedRMSE={metrics['debiased_rmse_km']:.4f} km"
            )

        run_manifest["status"] = "PASS"
        run_manifest["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        run_manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"\n🎉 实验全部完成，独立运行记录已保存至 Google Drive: {run_dir}")
    except Exception as exc:
        run_manifest["status"] = "FAIL"
        run_manifest["error"] = f"{type(exc).__name__}: {exc}"
        run_manifest["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        run_manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        raise

## 3. 实验结果汇总与独立配置对比（复用官方 `metrics_contract`）

直接复用纯标准库评估模块 `scripts.remote_workflow.metrics_contract`（与 `haiqin` 服务器端完全一致的 6 列预测 CSV 校验与指标定义），汇总：
1. Google Drive `DRIVE_EXP_ROOT` 下各独立实验目录的 `RMSE`、`MAE`、`Bias`、`Debiased RMSE`、`MAPE (%)`、`P95` 及关联的 `dataset_config_key` / `val_split_mode`；
2. 仓库本地 `output/data/` 中已同步的 `base_pack` 运行指标（若存在）。

In [ ]:
# Cell 4: 扫描 Google Drive 独立实验目录与仓库 output/data 基准运行
import csv
import json

from scripts.remote_workflow.metrics_contract import (
    compute_prediction_metrics,
    parse_prediction_file,
)

DRIVE_RUN_METRICS: list[dict] = []
drive_runs = sorted(p for p in DRIVE_EXP_ROOT.iterdir() if p.is_dir()) if DRIVE_EXP_ROOT.exists() else []

print(f"=== Google Drive 实验产出目录 ({DRIVE_EXP_ROOT}) : 共 {len(drive_runs)} 个运行目录 ===")
for run_dir in drive_runs:
    manifest_file = run_dir / "run_manifest.json"
    run_meta = json.loads(manifest_file.read_text(encoding="utf-8")) if manifest_file.is_file() else {}
    ds_key = run_meta.get("dataset_config_key", "legacy")
    val_mode = run_meta.get("effective_config", {}).get("training", {}).get("val_split_mode", "unknown")

    pred_csvs = sorted(p for p in run_dir.rglob("*_predictions_*.csv") if not p.name.startswith("._"))
    if not pred_csvs:
        print(f"- {run_dir.name} (状态={run_meta.get('status', '进行中/无预测CSV')}, dataset={ds_key})")
        continue

    for csv_path in pred_csvs:
        try:
            m = compute_prediction_metrics(parse_prediction_file(csv_path, check_abs_error=False))
        except ValueError as exc:
            print(f"- {run_dir.name}/{csv_path.name}: CSV 校验未通过 ({exc})")
            continue
        model_label = csv_path.parents[1].name if csv_path.parent.name == "test_outputs" else csv_path.stem
        row_label = f"{run_dir.name} [{model_label}]"
        DRIVE_RUN_METRICS.append({
            "run": run_dir.name,
            "label": row_label,
            "model": model_label,
            "dataset_config_key": ds_key,
            "val_split_mode": val_mode,
            **m,
        })
        print(
            f"- {row_label} | dataset={ds_key} | val={val_mode} | "
            f"RMSE={m['rmse_km']:.4f} km | DebiasedRMSE={m['debiased_rmse_km']:.4f} km | "
            f"MAE={m['mae_km']:.4f} km | Bias={m['bias_km']:+.4f} km | P95={m['p95_abs_km']:.4f} km (N={m['samples']})"
        )

OUTPUT_DATA = PROJECT_ROOT / "output" / "data"
ARCHIVE_SUMMARY: dict[str, list] = {"base_pack": []}
for metrics_csv in sorted(OUTPUT_DATA.glob("*-base-*/reports/visualization/metrics.csv")):
    with metrics_csv.open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r["method"] in ("RBD", "SCM") and "real_cnn" not in r["model"]:
                ARCHIVE_SUMMARY["base_pack"].append({"run": metrics_csv.parents[2].name, **r})
ARCHIVE_SUMMARY["base_pack"].sort(key=lambda r: float(r["rmse_km"]))

if ARCHIVE_SUMMARY["base_pack"]:
    print("\n[基准参考] output/data 中 base_pack 运行 Top-5 测试集 RMSE:")
    for r in ARCHIVE_SUMMARY["base_pack"][:5]:
        print(
            f"  {r['run'][:42]:42s} | {r['method']:3s} | {r['model']:22s} | "
            f"RMSE={float(r['rmse_km']):.4f} km | MAE={float(r['mae_km']):.4f} km"
        )

In [ ]:
# Cell 5: 绘制实验对比看板（Google Drive 独立实验与 output/data 基准运行）
has_plot_data = bool(DRIVE_RUN_METRICS or ARCHIVE_SUMMARY.get("base_pack"))
if importlib.util.find_spec("matplotlib") is None or not has_plot_data:
    print("ℹ️ 未检测到 matplotlib 或当前尚无已完成的实验/基准数据，跳过图形渲染（完成上方训练后重新运行本单元即可出图）。")
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    ax = axes[0]
    if DRIVE_RUN_METRICS:
        top = sorted(DRIVE_RUN_METRICS, key=lambda r: r["rmse_km"])[:10]
        labels = [f"{r['model']} ({r['dataset_config_key'][:18]})" for r in top][::-1]
        ax.barh(labels, [r["rmse_km"] for r in top][::-1], color="#2b6cb0")
        ax.set_title("Google Drive Runs Test RMSE (km)")
    else:
        top = ARCHIVE_SUMMARY["base_pack"][:8]
        ax.barh([f"{r['method']}-{r['model']}" for r in top][::-1], [float(r["rmse_km"]) for r in top][::-1], color="#2b6cb0")
        ax.set_title("base_pack output/data Top-8 Test RMSE (km)")
    ax.set_xlabel("RMSE (km)")

    ax = axes[1]
    if DRIVE_RUN_METRICS:
        top = sorted(DRIVE_RUN_METRICS, key=lambda r: abs(r["bias_km"]))[:10]
        labels = [f"{r['model']} ({r['dataset_config_key'][:18]})" for r in top][::-1]
        ax.barh(labels, [r["bias_km"] for r in top][::-1], color="#2f855a")
        ax.set_title("Google Drive Runs Signed Bias (km)")
    else:
        top = ARCHIVE_SUMMARY["base_pack"][:8]
        ax.barh([f"{r['method']}-{r['model']}" for r in top][::-1], [float(r["bias_km"]) for r in top][::-1], color="#2f855a")
        ax.set_title("base_pack output/data Signed Bias (km)")
    ax.axvline(0, color="#555555", lw=1)
    ax.set_xlabel("Bias = prediction - truth (km)")

    if IN_COLAB or DRIVE_EXP_ROOT.exists():
        DRIVE_EXP_ROOT.mkdir(parents=True, exist_ok=True)
        fig_out = DRIVE_EXP_ROOT / "swellex_benchmark_overview.png"
        fig.savefig(fig_out, dpi=160)
        print(f"📊 对比图已保存至: {fig_out}")
    plt.show()